# 5. Experiments — creative moves beyond `4_final_model.ipynb`

**Purpose:** scratchpad for genuinely different approaches that might beat the shippable pooled-XGBoost submission. Nothing here pollutes `4_final_model.ipynb` — promote an experiment into notebook 4 only if it clearly beats it on Jul holdout WMAPE **and** produces saner August predictions (especially for Portfolio A).

## Experiments in this notebook

1. **Adversarial validation (diagnostic, run first).** Does August look like the training data at all? If a classifier can distinguish Aug from train, our Jul-holdout metrics are lying about Aug performance. Run this before touching any model.
2. **Seasonal-naive baseline + learned ensemble blend.** The stupidest possible model: `aug_2025 = aug_2024 × recent_ratio`. Use it as an ensemble member and learn blend weights per portfolio. Cheap insurance against XGBoost's weird Portfolio A behavior.
3. **Erlang-A queueing model for Abandon Rate.** Mechanical call-center formula: given arrival rate (CV), service rate (1/CCT), agent count (staffing), and patience (learned), abandon rate is a closed-form function. Potentially unlocks the ABD problem that regression keeps struggling with.

## Baseline to beat
Load `submission_forecast.csv` from notebook 4 at the end and compare each experiment's August predictions against it + historical averages.


## 1. Setup — imports and config (mirrors notebook 4)

In [8]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from sklearn.metrics import roc_auc_score, mean_absolute_error
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

DATA_FILE = "Data for Datathon (Revised).xlsx"
TEMPLATE_FILE = "template_forecast_v00.csv"
PORTFOLIOS = ['A', 'B', 'C', 'D']
MONTH_MAP = {'January': 1, 'February': 2, 'March': 3, 'April': 4,
             'May': 5, 'June': 6, 'July': 7, 'August': 8,
             'September': 9, 'October': 10, 'November': 11, 'December': 12}

US_HOLIDAYS = set([
    datetime(2024, 1, 1), datetime(2024, 1, 15), datetime(2024, 2, 19),
    datetime(2024, 5, 27), datetime(2024, 7, 4), datetime(2024, 9, 2),
    datetime(2024, 10, 14), datetime(2024, 11, 11), datetime(2024, 11, 28),
    datetime(2024, 12, 25),
    datetime(2025, 1, 1), datetime(2025, 1, 20), datetime(2025, 2, 17),
    datetime(2025, 5, 26), datetime(2025, 7, 4), datetime(2025, 9, 1),
    datetime(2025, 10, 13), datetime(2025, 11, 11), datetime(2025, 11, 27),
    datetime(2025, 12, 25)
])


## 2. Load data (copied from notebook 4 cell 3)

In [9]:
daily_data = {}
for p in PORTFOLIOS:
    df = pd.read_excel(DATA_FILE, sheet_name=f"{p} - Daily")
    df['Date'] = pd.to_datetime(df['Date'].str.strip().str[:8], format='%m/%d/%y')
    df = df.sort_values('Date').reset_index(drop=True)
    df['dow'] = df['Date'].dt.dayofweek
    df['month'] = df['Date'].dt.month
    df['day_of_month'] = df['Date'].dt.day
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
    df['is_weekend'] = (df['dow'] >= 5).astype(int)
    df['is_holiday'] = df['Date'].isin(US_HOLIDAYS).astype(int)
    df['year'] = df['Date'].dt.year
    df['day_of_year'] = df['Date'].dt.dayofyear
    df['is_month_start'] = (df['day_of_month'] <= 3).astype(int)
    df['is_month_end'] = (df['day_of_month'] >= 28).astype(int)
    df['is_friday'] = (df['dow'] == 4).astype(int)
    df['is_monday'] = (df['dow'] == 0).astype(int)
    df['is_day_after_holiday'] = df['Date'].apply(
        lambda d: int((d - timedelta(days=1)) in US_HOLIDAYS))
    df['Abandoned Calls'] = df['Call Volume'] * df['Abandon Rate']
    daily_data[p] = df

staffing = pd.read_excel(DATA_FILE, sheet_name="Daily Staffing")
staffing.rename(columns={'Unnamed: 0': 'Date'}, inplace=True)
staffing['Date'] = pd.to_datetime(staffing['Date'], format='mixed')

print("Loaded.")
for p in PORTFOLIOS:
    print(f"  {p}: {len(daily_data[p])} daily ({daily_data[p]['Date'].min().date()} to {daily_data[p]['Date'].max().date()})")


Loaded.
  A: 731 daily (2024-01-01 to 2025-12-31)
  B: 731 daily (2024-01-01 to 2025-12-31)
  C: 731 daily (2024-01-01 to 2025-12-31)
  D: 731 daily (2024-01-01 to 2025-12-31)


## 3. Experiment #4 — Adversarial validation

**Question: does August look like our training data?**

Build a binary classifier where `label=1` if a row is an August day, `label=0` otherwise. If the classifier does meaningfully better than AUC=0.5, August is structurally different from training — and our Jul holdout metrics are *optimistic* relative to Aug performance.

**Features:** the calendar/seasonal features only (not lag features, since lag features are by definition different for August depending on what's lagged in — they'd leak the label).

**Interpretation:**
- **AUC ~0.50:** August is just a normal month, trust Jul holdout metrics.
- **AUC 0.55–0.65:** Mild shift. Consider weighting training by classifier probabilities.
- **AUC > 0.70:** Strong shift. Jul holdout is lying to you; weight training heavily toward Aug-like days, or accept submission variance.


In [10]:
# Stack all daily data into one frame, label Aug rows
rows = []
for p in PORTFOLIOS:
    d = daily_data[p].copy()
    d['portfolio'] = p
    rows.append(d)
all_daily = pd.concat(rows, ignore_index=True)

# Use only calendar/structural features (no targets, no lags — they'd leak)
adv_features = ['dow', 'month', 'day_of_month', 'week_of_year', 'is_weekend',
                'is_holiday', 'year', 'day_of_year', 'is_month_start', 'is_month_end',
                'is_friday', 'is_monday', 'is_day_after_holiday']

# Merge staffing as one more signal
staff_long = staffing.melt(id_vars='Date', value_vars=PORTFOLIOS,
                            var_name='portfolio', value_name='staffing')
all_daily = all_daily.merge(staff_long, on=['Date', 'portfolio'], how='left')
adv_features = adv_features + ['staffing']

# Label: 1 = August (either year), 0 = other months
all_daily['is_aug'] = (all_daily['month'] == 8).astype(int)

X_adv = all_daily[adv_features].fillna(-1).values
y_adv = all_daily['is_aug'].values

# Train a simple classifier with k-fold to avoid overfit optimism
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y_adv))
for tr, va in kf.split(X_adv):
    clf = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
    clf.fit(X_adv[tr], y_adv[tr])
    oof_preds[va] = clf.predict_proba(X_adv[va])[:, 1]

auc = roc_auc_score(y_adv, oof_preds)
print(f'Adversarial validation AUC (Aug vs not-Aug): {auc:.4f}')

if auc < 0.55:
    print('  → August is basically indistinguishable from training. Trust Jul holdout metrics.')
elif auc < 0.65:
    print('  → Mild distribution shift. Jul metrics are slightly optimistic for Aug.')
elif auc < 0.75:
    print('  → Meaningful shift. Consider upweighting Aug-like training days.')
else:
    print('  → Strong shift. Jul holdout is unreliable for Aug — weight training heavily.')

# Which features distinguish August?
clf_full = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42)
clf_full.fit(X_adv, y_adv)
importances = sorted(zip(adv_features, clf_full.feature_importances_), key=lambda x: -x[1])
print('\nTop discriminative features (Aug vs not-Aug):')
for f, imp in importances[:8]:
    print(f'  {f}: {imp:.4f}')


Adversarial validation AUC (Aug vs not-Aug): 1.0000
  → Strong shift. Jul holdout is unreliable for Aug — weight training heavily.

Top discriminative features (Aug vs not-Aug):
  month: 1.0000
  day_of_year: 0.0000
  dow: 0.0000
  week_of_year: 0.0000
  is_weekend: 0.0000
  is_holiday: 0.0000
  year: 0.0000
  is_month_start: 0.0000


### If AUC > 0.6, try this: classifier-weighted training

Take the OOF probabilities from above, use them as sample weights in the XGBoost training. Rows that "look like August" get higher weight, pulling the model toward Aug-relevant structure.

```python
# Outline (implement after reading AUC result above)
sample_weights = 0.5 + oof_preds  # scales to [0.5, 1.5]
# pass weight= to xgb.DMatrix when retraining
```


## 4. Experiment #2 — Seasonal-naive baseline + ensemble blend

The anti-ML move: `aug_2025[p][d] = aug_2024[p][d] × ratio`, where `ratio = (Jan–Jul 2025 mean) / (Jan–Jul 2024 mean)` per portfolio.

Then compare against notebook 4's XGBoost August predictions and fit a per-portfolio blend weight on Jul holdout:

```
blend = w × xgb + (1 − w) × seasonal_naive
```

If seasonal_naive is good enough, `w < 1` and we hedge the XGB variance.


In [11]:
def seasonal_naive_august(metric_col):
    """For each portfolio, project Aug 2024 forward using the Jan-Jul YoY ratio."""
    preds = {}
    for p in PORTFOLIOS:
        d = daily_data[p]
        # Mean over Jan-Jul per year
        h1_2024 = d[(d['year']==2024) & (d['month'].isin(range(1,8)))][metric_col].mean()
        h1_2025 = d[(d['year']==2025) & (d['month'].isin(range(1,8)))][metric_col].mean()
        ratio = h1_2025 / h1_2024 if h1_2024 > 0 else 1.0
        aug24 = d[(d['year']==2024) & (d['month']==8)].sort_values('day_of_month')
        aug25_proj = aug24[metric_col].values * ratio
        preds[p] = aug25_proj
        print(f'  {p} {metric_col}: ratio={ratio:.3f}, Aug24 mean={aug24[metric_col].mean():.1f}, '
              f'Aug25 proj mean={aug25_proj.mean():.1f}')
    return preds

print('=== Seasonal-naive projections for August 2025 ===')
print('\nCall Volume:')
sn_cv = seasonal_naive_august('Call Volume')
print('\nCCT:')
sn_cct = seasonal_naive_august('CCT')
print('\nAbandon Rate:')
sn_abd = seasonal_naive_august('Abandon Rate')

# Compare to notebook 4's predictions
print('\n=== Compare seasonal-naive vs notebook 4 ===')
try:
    sub4 = pd.read_csv('submission_forecast.csv')
    for p in PORTFOLIOS:
        nb4_cv = sub4.groupby('Day')[f'Calls_Offered_{p}'].sum().mean()
        print(f'  {p} CV: seasonal_naive={sn_cv[p].mean():.0f}, notebook4={nb4_cv:.0f}, '
              f'hist={daily_data[p]["Call Volume"].mean():.0f}')
except FileNotFoundError:
    print('  submission_forecast.csv not found — run notebook 4 first.')


=== Seasonal-naive projections for August 2025 ===

Call Volume:
  A Call Volume: ratio=0.903, Aug24 mean=4121.3, Aug25 proj mean=3720.1
  B Call Volume: ratio=0.867, Aug24 mean=9236.0, Aug25 proj mean=8010.0
  C Call Volume: ratio=0.900, Aug24 mean=20042.4, Aug25 proj mean=18042.5
  D Call Volume: ratio=0.884, Aug24 mean=11193.7, Aug25 proj mean=9890.7

CCT:
  A CCT: ratio=1.012, Aug24 mean=316.5, Aug25 proj mean=320.3
  B CCT: ratio=0.996, Aug24 mean=329.0, Aug25 proj mean=327.8
  C CCT: ratio=0.988, Aug24 mean=336.0, Aug25 proj mean=331.9
  D CCT: ratio=1.008, Aug24 mean=316.8, Aug25 proj mean=319.3

Abandon Rate:
  A Abandon Rate: ratio=0.907, Aug24 mean=0.0, Aug25 proj mean=0.0
  B Abandon Rate: ratio=0.950, Aug24 mean=0.0, Aug25 proj mean=0.0
  C Abandon Rate: ratio=0.932, Aug24 mean=0.0, Aug25 proj mean=0.0
  D Abandon Rate: ratio=1.099, Aug24 mean=0.0, Aug25 proj mean=0.0

=== Compare seasonal-naive vs notebook 4 ===
  A CV: seasonal_naive=3720, notebook4=3585, hist=4105
  B CV

### Blend weight search (stub)

After running the cell above, fit per-portfolio blend weights on the Jul 2025 holdout. For each portfolio:
1. Compute XGBoost Jul predictions (re-run notebook 4's Stage-1 or load the model).
2. Compute seasonal-naive Jul predictions (analogous to the Aug projection above, but targeting Jul using Jan–Jun ratio).
3. Sweep `w ∈ [0, 1]` in steps of 0.05, pick the w that minimizes Jul WMAPE on that portfolio.
4. Apply the chosen `w` to blend Aug predictions.

If `w ≈ 1` for all portfolios, seasonal-naive adds nothing and you can skip. If any portfolio picks `w < 0.8`, the blend is helping there.


In [12]:
# Blend weight search: for each portfolio, find w ∈ [0, 1] that minimizes
# Jul-holdout WMAPE on the (w × XGB + (1-w) × seasonal_naive) combination.
#
# We train a quick pooled XGBoost on the Jan-Jun training window so we can
# generate Jul holdout predictions inside this notebook (no dependency on NB4's
# fitted models being in memory).

from sklearn.linear_model import RidgeCV  # not used, kept for reference

# --- Build the same daily_df as notebook 4 ---
daily_metrics = ['Call Volume', 'CCT', 'Abandon Rate', 'Abandoned Calls']

def build_daily_features(df, staffing_df, portfolio_label, portfolio_id):
    d = df.copy()
    d['portfolio'] = portfolio_id
    for metric in daily_metrics:
        for lag in [1, 7, 14, 28, 365]:
            d[f'{metric}_lag{lag}'] = d[metric].shift(lag)
        for window in [7, 14, 28]:
            d[f'{metric}_roll{window}'] = d[metric].rolling(window).mean().shift(1)
            d[f'{metric}_rollstd{window}'] = d[metric].rolling(window).std().shift(1)
        d[f'{metric}_same_dow_1w'] = d[metric].shift(7)
        d[f'{metric}_same_dow_2w'] = d[metric].shift(14)
        d[f'{metric}_same_dow_avg'] = (d[metric].shift(7) + d[metric].shift(14)) / 2
    staff = staffing_df[['Date', portfolio_label]].copy()
    staff.columns = ['Date', 'staffing']
    d = d.merge(staff, on='Date', how='left')
    return d

all_daily_list = []
for i, p in enumerate(PORTFOLIOS):
    feat = build_daily_features(daily_data[p], staffing, p, i)
    all_daily_list.append(feat)
daily_df = pd.concat(all_daily_list, ignore_index=True)

base_features = ['dow','month','day_of_month','week_of_year','is_weekend','is_holiday',
                 'year','day_of_year','is_month_start','is_month_end','is_friday','is_monday',
                 'is_day_after_holiday','portfolio']
cv_features = base_features + [
    'Call Volume_lag1','Call Volume_lag7','Call Volume_lag14','Call Volume_lag28','Call Volume_lag365',
    'Call Volume_roll7','Call Volume_roll14','Call Volume_roll28','Call Volume_rollstd7',
    'Call Volume_same_dow_avg','staffing']
cct_features = base_features + [
    'CCT_lag7','CCT_lag14','CCT_lag28','CCT_lag365',
    'CCT_roll7','CCT_roll14','CCT_roll28','CCT_same_dow_avg','staffing']
abd_features = base_features + [
    'Abandoned Calls_lag7','Abandoned Calls_lag14','Abandoned Calls_lag28','Abandoned Calls_lag365',
    'Abandoned Calls_roll7','Abandoned Calls_roll14','Abandoned Calls_roll28',
    'Abandoned Calls_same_dow_avg',
    'Call Volume_lag7','Call Volume_lag14','Call Volume_roll7','staffing']

DAILY_TARGETS = {'cv': 'Call Volume', 'cct': 'CCT', 'abd': 'Abandoned Calls'}
DAILY_FEATURES = {'cv': cv_features, 'cct': cct_features, 'abd': abd_features}

train_mask = daily_df['Date'] < datetime(2025, 7, 1)
val_mask   = (daily_df['Date'] >= datetime(2025, 7, 1)) & (daily_df['Date'] < datetime(2025, 8, 1))

# --- Train quick pooled XGB per metric (default-ish params, no Optuna — just a speed stand-in) ---
quick_params = {'cv':  {'objective':'reg:squarederror', 'max_depth':6, 'learning_rate':0.05,
                        'subsample':0.8, 'colsample_bytree':0.8, 'enable_categorical':True,
                        'verbosity':0, 'seed':42},
                'cct': {'objective':'reg:squarederror', 'max_depth':6, 'learning_rate':0.05,
                        'subsample':0.8, 'colsample_bytree':0.8, 'enable_categorical':True,
                        'verbosity':0, 'seed':42},
                'abd': {'objective':'count:poisson',    'max_depth':6, 'learning_rate':0.05,
                        'subsample':0.8, 'colsample_bytree':0.8, 'enable_categorical':True,
                        'verbosity':0, 'seed':42}}

xgb_jul_preds = {m: None for m in DAILY_TARGETS}   # keyed by metric, value = dict[pid] -> np.array
xgb_jul_actuals = {m: None for m in DAILY_TARGETS}

for mk, tgt in DAILY_TARGETS.items():
    feats = DAILY_FEATURES[mk]
    X_tr = daily_df[train_mask][feats].copy()
    y_tr = daily_df[train_mask][tgt].copy()
    X_va = daily_df[val_mask][feats].copy()
    y_va = daily_df[val_mask][tgt].copy()
    v_tr, v_va = y_tr.notna(), y_va.notna()
    X_tr, y_tr = X_tr[v_tr], y_tr[v_tr]
    X_va, y_va = X_va[v_va], y_va[v_va]
    pf_va = daily_df[val_mask]['portfolio'].values[v_va.values]
    X_tr['portfolio'] = X_tr['portfolio'].astype('category')
    X_va['portfolio'] = X_va['portfolio'].astype('category')

    p = dict(quick_params[mk])
    ym = float(y_tr.mean())
    p['base_score'] = max(ym, 1e-3) if np.isfinite(ym) and ym > 0 else 1.0
    dtr = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
    dva = xgb.DMatrix(X_va, label=y_va, enable_categorical=True)
    m = xgb.train(p, dtr, num_boost_round=300, evals=[(dva,'va')],
                  early_stopping_rounds=30, verbose_eval=False)
    pred = np.clip(m.predict(dva), 0, None)

    per_p_pred = {pid: pred[pf_va == pid] for pid in range(len(PORTFOLIOS))}
    per_p_act  = {pid: y_va.values[pf_va == pid] for pid in range(len(PORTFOLIOS))}
    xgb_jul_preds[mk] = per_p_pred
    xgb_jul_actuals[mk] = per_p_act

# --- Seasonal-naive Jul projections (mirror of the Aug projection logic) ---
def seasonal_naive_jul(metric_col):
    out = {}
    for pid, p in enumerate(PORTFOLIOS):
        d = daily_data[p]
        h1a_2024 = d[(d['year']==2024) & (d['month'].isin(range(1,7)))][metric_col].mean()
        h1a_2025 = d[(d['year']==2025) & (d['month'].isin(range(1,7)))][metric_col].mean()
        ratio = h1a_2025 / h1a_2024 if h1a_2024 > 0 else 1.0
        jul24 = d[(d['year']==2024) & (d['month']==7)].sort_values('day_of_month')[metric_col].values
        out[pid] = jul24 * ratio
    return out

sn_jul = {'cv':  seasonal_naive_jul('Call Volume'),
          'cct': seasonal_naive_jul('CCT'),
          'abd': seasonal_naive_jul('Abandoned Calls')}

# --- Grid search blend weight per (metric, portfolio) ---
def wmape(actual, pred):
    actual, pred = np.asarray(actual), np.asarray(pred)
    denom = np.sum(np.abs(actual))
    return np.sum(np.abs(actual - pred)) / denom if denom > 0 else float('nan')

print(f'{"metric":<6} {"port":<6} {"xgb_wmape":>11} {"naive_wmape":>12} {"best_w":>8} {"blend_wmape":>12}')
print('-' * 60)
blend_weights = {}
for mk in DAILY_TARGETS:
    blend_weights[mk] = {}
    for pid, p in enumerate(PORTFOLIOS):
        xgb_pred = xgb_jul_preds[mk][pid]
        act      = xgb_jul_actuals[mk][pid]
        naive    = sn_jul[mk][pid]
        # Align lengths — seasonal-naive gives Jul 2024 length (31 days),
        # XGB Jul predictions are however many valid Jul 2025 rows exist
        L = min(len(xgb_pred), len(act), len(naive))
        if L == 0:
            continue
        xgb_pred, act, naive = xgb_pred[:L], act[:L], naive[:L]
        w_xgb = wmape(act, xgb_pred)
        w_nai = wmape(act, naive)
        best_w, best_wmape = 1.0, w_xgb
        for w in np.arange(0, 1.01, 0.05):
            blend = w * xgb_pred + (1-w) * naive
            wm = wmape(act, blend)
            if wm < best_wmape:
                best_w, best_wmape = w, wm
        blend_weights[mk][pid] = best_w
        print(f'{mk:<6} {p:<6} {w_xgb:>11.4f} {w_nai:>12.4f} {best_w:>8.2f} {best_wmape:>12.4f}')

print('\nPicked weights (lower w = more seasonal-naive contribution):')
print(blend_weights)


metric port     xgb_wmape  naive_wmape   best_w  blend_wmape
------------------------------------------------------------
cv     A           0.0805       0.2787     0.95       0.0786
cv     B           0.0792       0.2399     0.95       0.0765
cv     C           0.0387       0.1922     0.95       0.0379
cv     D           0.0454       0.2644     0.95       0.0436
cct    A           0.0296       0.0299     0.50       0.0255
cct    B           0.0193       0.0308     0.80       0.0184
cct    C           0.0167       0.0220     0.80       0.0163
cct    D           0.0181       0.0317     0.75       0.0172
abd    A           0.4794       0.6564     0.75       0.4468
abd    B           0.3517       0.4513     0.65       0.2940
abd    C           0.5801       1.0762     0.95       0.5776
abd    D           0.6616       1.0598     0.85       0.6332

Picked weights (lower w = more seasonal-naive contribution):
{'cv': {0: 0.9500000000000001, 1: 0.9500000000000001, 2: 0.9500000000000001, 3: 0.95

## 5. Experiment #1 — Erlang-A for Abandon Rate

**The insight:** abandon rate isn't a free-floating regression target. It's a *derived quantity* from how many agents are handling calls, how fast they handle them, and how patient callers are. Queueing theory gives us the exact formula (Erlang-A, a.k.a. Erlang with impatience).

### Setup
For each 30-min interval:
- `λ` = arrival rate (calls per unit time) — comes from Stage-1 CV prediction / 30 min
- `μ` = service rate (1 / mean handle time) — comes from Stage-1 CCT prediction
- `N` = number of agents active — comes from daily staffing / number of intervals agents work
- `θ` = caller patience rate (1 / mean time-to-abandon) — **the one parameter we fit**

Erlang-A abandon probability (simplified, M/M/N + M):
```
P(abandon) ≈ (λ / (N·μ)) ^ N  / (1 + slack(θ))    # handwave form
```

Full formula involves an Erlang-B term; use `scipy.stats` or implement the recursion directly. Patience θ is fit by matching historical daily abandon rates.

### Why this could unlock the problem
- The CV × staffing interaction that notebook 3 §3 flagged becomes **exact**, not learned.
- Zero training compute — fit one θ per portfolio from historical data.
- Automatically satisfies `0 ≤ abandon_rate ≤ 1` and scales correctly with volume.
- Transfers across portfolios even when data is thin.


In [13]:
# Erlang-A / Erlang-C implementation (simplified, single-class queue)
# Uses the Palm formula for Erlang-C + patience correction.

def erlang_c(lam, mu, N):
    """Probability of queueing in M/M/N (Erlang-C)."""
    if N <= 0 or mu <= 0:
        return 1.0
    rho = lam / (N * mu)
    if rho >= 1:
        return 1.0  # unstable queue
    # Erlang-B recursion
    a = lam / mu  # offered load in Erlangs
    B = 1.0
    for k in range(1, N + 1):
        B = (a * B) / (k + a * B)
    C = B / (1 - rho * (1 - B))
    return C

def erlang_a_abandon(lam, mu, N, theta):
    """
    Approximate abandon probability in M/M/N+M (Erlang-A).
    lam: arrival rate (calls/unit time)
    mu:  service rate (1/mean handle time)
    N:   number of servers (agents)
    theta: abandonment rate (1/mean patience time)
    Returns: P(abandon)
    """
    if lam <= 0:
        return 0.0
    if N <= 0:
        return 1.0
    C = erlang_c(lam, mu, N)
    rho = lam / (N * mu)
    if rho >= 1:
        # Unstable queue — abandonment approaches rho - 1 (not a formal result, proxy)
        return min(1.0 - 1.0 / rho, 0.99)
    # Approximation: abandon rate ≈ C × θ × E[wait]
    # E[wait | queued] ≈ 1 / (N·μ − λ) in M/M/N, adjusted for patience
    wait = 1.0 / (N * mu - lam + theta)
    p_abandon = C * (1 - np.exp(-theta * wait))
    return float(min(max(p_abandon, 0.0), 1.0))

# Quick sanity: 100 calls/hr, 5 min handle, 10 agents, θ = 1/120s = 0.0083/s
# Need consistent units. Use per-minute: lam=100/60, mu=1/5, N=10, theta=1/2
print('Sanity check:')
print(f'  100 calls/hr, 5-min handle, 10 agents, 2-min patience: '
      f'P(abandon)={erlang_a_abandon(100/60, 1/5, 10, 1/2):.4f}')
print(f'  100 calls/hr, 5-min handle, 20 agents, 2-min patience: '
      f'P(abandon)={erlang_a_abandon(100/60, 1/5, 20, 1/2):.4f}')
print(f'  100 calls/hr, 5-min handle,  5 agents, 2-min patience: '
      f'P(abandon)={erlang_a_abandon(100/60, 1/5,  5, 1/2):.4f}')


Sanity check:
  100 calls/hr, 5-min handle, 10 agents, 2-min patience: P(abandon)=0.2200
  100 calls/hr, 5-min handle, 20 agents, 2-min patience: P(abandon)=0.0001
  100 calls/hr, 5-min handle,  5 agents, 2-min patience: P(abandon)=0.4000


### Fitting patience θ per portfolio

For each portfolio, use historical daily data:
- `λ_day` = total calls / active hours (approximate active window ~12h)
- `μ_day` = 1 / CCT (in same units)
- `N_day` = staffing / shifts_per_day (assume 1.5 shifts, tweak)
- Observed abandon rate = `Abandon Rate` column

Fit θ by minimizing squared error between predicted and observed daily abandon rate across all valid training days.

```python
from scipy.optimize import minimize_scalar

def fit_patience(df_p, active_hours=12):
    def loss(theta):
        errs = []
        for _, row in df_p.iterrows():
            if pd.isna(row['Call Volume']) or pd.isna(row['CCT']) or pd.isna(row['staffing']):
                continue
            lam = row['Call Volume'] / (active_hours * 60)   # calls per minute
            mu  = 60 / row['CCT']                              # calls handled per min per agent
            N   = row['staffing']
            p_pred = erlang_a_abandon(lam, mu, N, theta)
            errs.append((p_pred - row['Abandon Rate']) ** 2)
        return np.mean(errs) if errs else 1e9
    result = minimize_scalar(loss, bounds=(1/600, 1/10), method='bounded')  # patience 10s-10min
    return result.x, result.fun
```

Then apply the fitted θ per portfolio to Aug predictions (λ, μ from XGBoost Stage-1; N from staffing).


In [14]:
from scipy.optimize import minimize_scalar

# Merge staffing into each portfolio's daily frame for easier iteration
daily_staff = {}
for pid, p in enumerate(PORTFOLIOS):
    d = daily_data[p].copy()
    staff = staffing[['Date', p]].rename(columns={p: 'staffing'})
    d = d.merge(staff, on='Date', how='left')
    daily_staff[p] = d

# --- Fit caller patience θ per portfolio by matching historical daily abandon rate ---
ACTIVE_HOURS = 12  # active call window per day (tune if known)

def fit_patience(df_p):
    """Find θ (1/mean patience in minutes) minimizing squared error on daily abandon rate."""
    def loss(theta):
        errs = []
        for _, row in df_p.iterrows():
            if any(pd.isna(row[c]) for c in ['Call Volume','CCT','staffing','Abandon Rate']):
                continue
            if row['staffing'] <= 0 or row['CCT'] <= 0:
                continue
            lam = row['Call Volume'] / (ACTIVE_HOURS * 60)   # calls per minute
            mu  = 60 / row['CCT']                             # calls per minute per agent (CCT in seconds)
            N   = max(int(round(row['staffing'])), 1)
            p_pred = erlang_a_abandon(lam, mu, N, theta)
            errs.append((p_pred - row['Abandon Rate']) ** 2)
        return np.mean(errs) if errs else 1e9
    # Patience range: 10 seconds to 20 minutes → θ ∈ [1/20, 1/(10/60)] = [0.05, 6] per minute
    result = minimize_scalar(loss, bounds=(0.05, 6.0), method='bounded',
                              options={'xatol': 1e-3})
    return result.x, result.fun

print('=== Fitting caller patience θ per portfolio ===')
fitted_theta = {}
for p in PORTFOLIOS:
    theta_star, loss_star = fit_patience(daily_staff[p])
    fitted_theta[p] = theta_star
    mean_patience_min = 1.0 / theta_star
    print(f'  {p}: θ*={theta_star:.4f} per min (mean patience {mean_patience_min:.2f} min), '
          f'loss={loss_star:.6f}')

# --- Apply Erlang-A to August predictions ---
# Pull Stage-1 CV and CCT daily predictions from notebook 4's submission (aggregate intervals to daily)
try:
    sub4 = pd.read_csv('submission_forecast.csv')
except FileNotFoundError:
    print('\nsubmission_forecast.csv not found — run notebook 4 first.')
    sub4 = None

if sub4 is not None:
    print('\n=== Erlang-A August abandon rate vs notebook 4 ===')
    print(f'{"port":<6} {"erlang_A_rate":>14} {"nb4_rate":>11} {"hist_rate":>11}')
    print('-' * 46)
    erlang_aug_rates = {}
    for pid, p in enumerate(PORTFOLIOS):
        # Daily CV and CCT from NB4 submission (already contains per-day per-portfolio values)
        daily_cv_aug = sub4.groupby('Day')[f'Calls_Offered_{p}'].sum().values
        daily_cct_aug = sub4.groupby('Day')[f'CCT_{p}'].mean().values  # mean of interval CCTs

        # Staffing for Aug days — use historical same-DOW fallback if Aug isn't in the sheet
        aug_staff = []
        for day in range(1, 32):
            ad = datetime(2025, 8, day)
            row = staffing[staffing['Date'] == ad]
            if len(row) > 0 and pd.notna(row[p].values[0]):
                aug_staff.append(float(row[p].values[0]))
            else:
                same_dow = staffing[staffing['Date'].dt.weekday == ad.weekday()][p].dropna()
                aug_staff.append(float(same_dow.iloc[-1]) if len(same_dow) > 0 else np.nan)
        aug_staff = np.array(aug_staff)

        theta_p = fitted_theta[p]
        rates = []
        for day_idx in range(len(daily_cv_aug)):
            cv_d, cct_d, N_d = daily_cv_aug[day_idx], daily_cct_aug[day_idx], aug_staff[day_idx]
            if not all(np.isfinite([cv_d, cct_d, N_d])) or N_d <= 0 or cct_d <= 0:
                rates.append(np.nan)
                continue
            lam = cv_d / (ACTIVE_HOURS * 60)
            mu  = 60 / cct_d
            r   = erlang_a_abandon(lam, mu, max(int(round(N_d)), 1), theta_p)
            rates.append(r)
        erlang_aug_rates[p] = np.array(rates)

        # NB4 weighted rate from submission
        cv_int = sub4[f'Calls_Offered_{p}']
        abd_int = sub4[f'Abandoned_Rate_{p}']
        nb4_rate = np.average(abd_int, weights=cv_int) if cv_int.sum() > 0 else 0.0
        hist_rate = daily_data[p]['Abandon Rate'].mean()
        erlang_mean = np.nanmean(erlang_aug_rates[p])
        print(f'{p:<6} {erlang_mean:>14.4f} {nb4_rate:>11.4f} {hist_rate:>11.4f}')

    print('\nInterpretation:')
    print('  - If erlang_A_rate is closer to hist_rate than nb4_rate → promote Erlang-A for ABD.')
    print('  - If erlang_A_rate is wildly off → ACTIVE_HOURS or staffing-to-agents assumption is wrong.')
    print('  - Re-tune ACTIVE_HOURS (try 8, 10, 14) and staffing interpretation if numbers look off.')


=== Fitting caller patience θ per portfolio ===
  A: θ*=0.0506 per min (mean patience 19.75 min), loss=0.000194
  B: θ*=0.0506 per min (mean patience 19.75 min), loss=0.061313
  C: θ*=5.9994 per min (mean patience 0.17 min), loss=0.000534
  D: θ*=5.9994 per min (mean patience 0.17 min), loss=0.000534

=== Erlang-A August abandon rate vs notebook 4 ===
port    erlang_A_rate    nb4_rate   hist_rate
----------------------------------------------
A              0.0040      0.0146      0.0113
B              0.0565      0.0190      0.0170
C              0.0000      0.0175      0.0124
D              0.0000      0.0181      0.0131

Interpretation:
  - If erlang_A_rate is closer to hist_rate than nb4_rate → promote Erlang-A for ABD.
  - If erlang_A_rate is wildly off → ACTIVE_HOURS or staffing-to-agents assumption is wrong.
  - Re-tune ACTIVE_HOURS (try 8, 10, 14) and staffing interpretation if numbers look off.


## 6. Summary — promotion decisions

Fill this in after running experiments 3/4/5 above.

| Experiment | Result | Worth promoting to notebook 4? |
|---|---|---|
| Adversarial validation (AUC) | *fill in* | Diagnostic only — not promoted |
| Seasonal-naive blend (picked weights) | *fill in* | *yes/no* |
| Erlang-A for ABD (fitted θ per portfolio) | *fill in* | *yes/no* |

### Decision rule
Promote an experiment into `4_final_model.ipynb` only if it:
1. Beats notebook 4 on Jul holdout WMAPE on ≥3 of 4 portfolios for the relevant metric, **and**
2. Produces August predictions closer to historical averages (especially for Portfolio A).

Otherwise leave it here as documented exploration.
